# Encoder, predictor and JEPA exports

Purpose: shared visible-patch encoders, optimized SDPA baseline, conventional positional predictor and stopped-gradient EMA teacher.
Inputs: BCHW images and original patch-index masks; target encoder consumes the complete image intentionally.
Outputs: ordered feature tensors and JEPA loss/metrics. Runtime: definitions only; under one second after imports.
Checkpoint location: orchestrator-owned `checkpoints/<run_id>/`; no training is run by importing this notebook.
Resume: run all definition cells top to bottom after installing the tagged-notebook importer; restore complete JEPA and optimizer state through the training module before the next optimizer step. Imports execute only `ahr-export` cells.

Each residual block is `x + mixer(LN(x))`, followed by `x + MLP(LN(x))`. The FMM encoder never invokes attention. Attention is used explicitly in the ViT baseline and predictor. Padding is zeroed after every residual block and excluded from all messages.


In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
"""FMM and optimized-attention encoders, positional predictor, and EMA JEPA."""
import copy
import math
import torch
from torch import nn
from torch.nn import functional as F


def patch_coordinates(indices, grid, dtype=torch.float32):
    """Original patch centers in [0,1]^2, x then y; shape [...,2]."""
    rows, cols = map(int, grid)
    return torch.stack(((indices.remainder(cols).to(dtype) + 0.5) / cols,
                        (indices.div(cols, rounding_mode="floor").to(dtype) + 0.5) / rows), dim=-1)


def position_features(indices, grid, dim, dtype=torch.float32):
    """Fixed Fourier features of original 2-D centers; arbitrary rectangular grids."""
    xy = patch_coordinates(indices, grid, dtype=torch.float32)
    frequencies = torch.exp(torch.linspace(0, math.log(1000), max(1, math.ceil(dim / 4)), device=indices.device))
    phase = 2 * math.pi * xy.unsqueeze(-1) * frequencies
    return torch.cat((phase.sin(), phase.cos()), dim=-1).flatten(-2)[..., :dim].to(dtype)


def validate_positions(indices, valid, batch_size, count):
    if indices.ndim != 2 or indices.shape[0] != batch_size or indices.shape != valid.shape:
        raise ValueError("indices/valid must have matching [B,V] shapes")
    if indices.dtype != torch.long or valid.dtype != torch.bool:
        raise TypeError("indices must be long and valid must be bool")
    if not bool(valid.any(dim=1).all()):
        raise ValueError("each sample requires at least one valid position")
    active = indices[valid]
    if bool(((active < 0) | (active >= count)).any()):
        raise ValueError("valid position outside original patch grid")
    for i, mask in zip(indices, valid):
        if i[mask].unique().numel() != int(mask.sum()):
            raise ValueError("valid positions must be unique within each sample/block")


class AttentionMixer(nn.Module):
    """PyTorch SDPA baseline; padded keys masked and padded query outputs zeroed."""
    def __init__(self, dim, heads=4):
        super().__init__()
        if dim % heads:
            raise ValueError("attention dimension must be divisible by heads")
        self.dim, self.heads = dim, heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.projection = nn.Linear(dim, dim)
        self.backend = "torch.nn.functional.scaled_dot_product_attention; runtime dispatch"

    def forward(self, x, indices=None, valid=None, grid=None):
        batch, count, dim = x.shape
        if valid is None:
            valid = torch.ones((batch, count), dtype=torch.bool, device=x.device)
        x = torch.where(valid.unsqueeze(-1), x, torch.zeros_like(x))
        qkv = self.qkv(x).reshape(batch, count, 3, self.heads, dim // self.heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=valid[:, None, None, :], dropout_p=0.0)
        out = out.transpose(1, 2).reshape(batch, count, dim)
        return self.projection(out) * valid.unsqueeze(-1)


def observed_attention_backend(device="cpu", dtype=torch.float32, dim=32, heads=4, tokens=16):
    """Profile actual dispatcher events for a representative SDPA call; no guessing."""
    device = torch.device(device)
    activities = [torch.profiler.ProfilerActivity.CPU]
    if device.type == "cuda":
        activities.append(torch.profiler.ProfilerActivity.CUDA)
    module = AttentionMixer(dim, heads).to(device=device, dtype=dtype)
    x = torch.randn(1, tokens, dim, device=device, dtype=dtype)
    with torch.profiler.profile(activities=activities) as prof:
        with torch.no_grad():
            module(x)
        if device.type == "cuda":
            torch.cuda.synchronize(device)
    events = sorted({evt.key for evt in prof.key_averages() if "attention" in evt.key.lower()})
    joined = " ".join(events)
    if "flash_attention_for_cpu" in joined:
        backend = "pytorch_flash_attention_for_cpu"
    elif "flash_attention" in joined:
        backend = "pytorch_flash_attention_cuda"
    elif "efficient_attention" in joined:
        backend = "pytorch_memory_efficient_attention"
    elif "cudnn_attention" in joined:
        backend = "pytorch_cudnn_attention"
    elif "attention_math" in joined:
        backend = "pytorch_sdpa_math"
    else:
        backend = "unclassified_sdpa_dispatch"
    return {"backend": backend, "events": events, "device": str(device), "dtype": str(dtype),
            "shape": [1, tokens, dim], "heads": heads}


class EncoderBlock(nn.Module):
    def __init__(self, kind, dim, heads=4, order=2, leaf_size=2, receiver_gate=True):
        super().__init__()
        if kind == "fmm":
            from fmm_jepa.operators import FMMMixer
            self.mixer = FMMMixer(dim, order=order, leaf_size=leaf_size, receiver_gate=receiver_gate)
        elif kind == "vit":
            self.mixer = AttentionMixer(dim, heads)
        else:
            raise ValueError("kind must be fmm or vit")
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, x, indices, valid, grid):
        x = x + self.mixer(self.norm1(x), indices, valid, grid)
        x = x + self.mlp(self.norm2(x))
        return x * valid.unsqueeze(-1)


class VisionEncoder(nn.Module):
    """Patch encoder returning [B,V,D] in exactly the supplied original-index order."""
    def __init__(self, kind="fmm", patch_size=8, dim=64, depth=2, heads=4,
                 order=2, leaf_size=2, receiver_gate=True):
        super().__init__()
        if patch_size < 1 or dim < 1 or depth < 1:
            raise ValueError("positive patch_size, dim and depth required")
        self.patch_size, self.dim, self.kind = patch_size, dim, kind
        self.patch_embed = nn.Linear(3 * patch_size * patch_size, dim)
        self.blocks = nn.ModuleList([EncoderBlock(kind, dim, heads, order, leaf_size, receiver_gate) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)

    def forward(self, images, indices=None, valid=None):
        if images.ndim != 4 or images.shape[1] != 3:
            raise ValueError("images must have shape [B,3,H,W]")
        batch, _, height, width = images.shape
        patch = self.patch_size
        if height % patch or width % patch:
            raise ValueError("image dimensions must be exactly divisible by patch_size")
        grid = (height // patch, width // patch)
        count = grid[0] * grid[1]
        if indices is None:
            if valid is not None:
                raise ValueError("valid cannot be supplied without indices")
            indices = torch.arange(count, device=images.device).expand(batch, -1)
        if valid is None:
            valid = torch.ones_like(indices, dtype=torch.bool)
        if indices.device != images.device or valid.device != images.device:
            raise ValueError("images, indices, and valid must share a device")
        validate_positions(indices, valid, batch, count)
        # Unfold raw pixels, then gather BEFORE any learned embedding/mixing.
        # This allocates O(B*N_grid*patch^2*3) raw data but no hidden feature tokens.
        raw = F.unfold(images, kernel_size=patch, stride=patch).transpose(1, 2)
        gather = indices.clamp(0, count - 1).unsqueeze(-1).expand(-1, -1, raw.shape[-1])
        visible_pixels = torch.where(valid.unsqueeze(-1), raw.gather(1, gather), torch.zeros_like(raw.gather(1, gather)))
        x = self.patch_embed(visible_pixels)
        x = (x + position_features(indices.clamp_min(0), grid, self.dim, x.dtype)) * valid.unsqueeze(-1)
        for block in self.blocks:
            x = block(x, indices, valid, grid)
        return self.norm(x) * valid.unsqueeze(-1)


class Predictor(nn.Module):
    """Conventional small Transformer over context and positional target mask tokens."""
    def __init__(self, encoder_dim, dim=32, depth=1, heads=4):
        super().__init__()
        self.dim = dim
        self.context_projection = nn.Linear(encoder_dim, dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, dim))
        nn.init.normal_(self.mask_token, std=0.02)
        self.blocks = nn.ModuleList([EncoderBlock("vit", dim, heads, 2, 2, True) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.output_projection = nn.Linear(dim, encoder_dim)

    def forward(self, context_features, context_indices, context_valid, target_indices, target_valid, grid):
        batch = context_features.shape[0]
        validate_positions(target_indices, target_valid, batch, grid[0] * grid[1])
        c = self.context_projection(context_features)
        c = (c + position_features(context_indices.clamp_min(0), grid, self.dim, c.dtype)) * context_valid.unsqueeze(-1)
        t = self.mask_token.expand(batch, target_indices.shape[1], -1)
        t = (t + position_features(target_indices.clamp_min(0), grid, self.dim, c.dtype)) * target_valid.unsqueeze(-1)
        z = torch.cat((c, t), dim=1)
        valid = torch.cat((context_valid, target_valid), dim=1)
        indices = torch.cat((context_indices, target_indices), dim=1)
        for block in self.blocks:
            z = block(z, indices, valid, grid)
        return self.output_projection(self.norm(z[:, c.shape[1]:])) * target_valid.unsqueeze(-1)


def normalized_jepa_loss(predictions, targets, valid_masks):
    """SmoothL1 beta=1, mean features+valid positions per sample, then batch+blocks."""
    if not predictions or len(predictions) != len(targets) or len(predictions) != len(valid_masks):
        raise ValueError("nonempty matching target block lists required")
    block_losses = []
    for prediction, target, valid in zip(predictions, targets, valid_masks):
        if prediction.shape != target.shape or prediction.shape[:2] != valid.shape or not bool(valid.any(-1).all()):
            raise ValueError("target/prediction/mask shape mismatch or empty block")
        element = F.smooth_l1_loss(prediction, target.detach(), beta=1.0, reduction="none")
        per_sample = (element * valid.unsqueeze(-1)).sum(dim=(1, 2)) / (valid.sum(-1) * prediction.shape[-1])
        block_losses.append(per_sample.mean())
    return torch.stack(block_losses).mean()


def _representation_metrics(features, valid, prefix):
    values = features.detach()[valid]
    return {prefix + "_norm": float(values.norm(dim=-1).mean()),
            prefix + "_std": float(values.float().std(dim=0, unbiased=False).mean())}


class JEPA(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = copy.deepcopy(config)
        keys = ("kind", "patch_size", "dim", "depth", "heads", "order", "leaf_size", "receiver_gate")
        self.context = VisionEncoder(**{key: config[key] for key in keys if key in config})
        self.target = copy.deepcopy(self.context).requires_grad_(False).eval()
        self.predictor = Predictor(self.context.dim, dim=config.get("predictor_dim", 32),
                                   depth=config.get("predictor_depth", 1),
                                   heads=config.get("predictor_heads", config.get("heads", 4)))

    def train(self, mode=True):
        super().train(mode)
        self.target.eval()
        return self

    @torch.no_grad()
    def update_teacher(self, momentum):
        if not 0 <= momentum <= 1:
            raise ValueError("EMA momentum must be in [0,1]")
        context_parameters = dict(self.context.named_parameters())
        for name, target in self.target.named_parameters():
            target.mul_(momentum).add_(context_parameters[name], alpha=1 - momentum)
        context_buffers = dict(self.context.named_buffers())
        for name, target in self.target.named_buffers():
            source = context_buffers[name]
            if target.is_floating_point() or target.is_complex():
                target.mul_(momentum).add_(source, alpha=1 - momentum)
            else:
                target.copy_(source)
        self.target.eval()

    def forward(self, images, context_indices, context_valid, target_indices, target_valid):
        target_indices = [target_indices] if isinstance(target_indices, torch.Tensor) else list(target_indices)
        target_valid = [target_valid] if isinstance(target_valid, torch.Tensor) else list(target_valid)
        if not target_indices or len(target_indices) != len(target_valid):
            raise ValueError("nonempty matching target index and valid lists required")
        grid = (images.shape[-2] // self.context.patch_size, images.shape[-1] // self.context.patch_size)
        for indices, valid in zip(target_indices, target_valid):
            validate_positions(indices, valid, images.shape[0], grid[0] * grid[1])
            for c, cv, t, tv in zip(context_indices, context_valid, indices, valid):
                if bool(torch.isin(c[cv], t[tv]).any()):
                    raise ValueError("context positions must exclude the union of all target blocks")
        context_features = self.context(images, context_indices, context_valid)
        with torch.no_grad():
            complete_target = self.target(images)
            normalized_target = F.layer_norm(complete_target, (complete_target.shape[-1],))
            targets = [normalized_target.gather(1, torch.where(v, i, 0).unsqueeze(-1).expand(-1, -1, self.context.dim)) * v.unsqueeze(-1)
                       for i, v in zip(target_indices, target_valid)]
        predictions = [self.predictor(context_features, context_indices, context_valid, indices, valid, grid)
                       for indices, valid in zip(target_indices, target_valid)]
        loss = normalized_jepa_loss(predictions, targets, target_valid)
        metrics = {"loss": float(loss.detach())}
        metrics.update(_representation_metrics(context_features, context_valid, "context"))
        metrics.update(_representation_metrics(torch.cat(targets, dim=1), torch.cat(target_valid, dim=1), "target"))
        metrics.update(_representation_metrics(torch.cat(predictions, dim=1), torch.cat(target_valid, dim=1), "prediction"))
        branch_stats = [getattr(block.mixer, "last_metrics", {}) for block in self.context.blocks]
        for name in ("near_norm", "far_norm"):
            values = [float(stats[name]) for stats in branch_stats if name in stats]
            if values:
                metrics[name] = sum(values) / len(values)
        return {"loss": loss, "prediction": predictions, "target": targets,
                "context_features": context_features, "metrics": metrics}
